GSO Feature Selection Model

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# === Imports ===
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# === Define 10 Fast ML Models (no Random Forest) ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "SGD": SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42),
    "Ridge": RidgeClassifier(class_weight="balanced", random_state=42),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "GaussianNB": GaussianNB(),
    "LinearSVC": LinearSVC(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1,
        random_state=42
    )
}

# Train all models
trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model


# === SHAP Analysis Function ===
def run_shap_analysis(model, model_name, X_train, X_test, y_test, feature_names):
    print(f"\n🔍 Running SHAP analysis for {model_name}...")

    # Create SHAP Explainer
    try:
        explainer = shap.Explainer(model, X_train, feature_names=feature_names)
    except Exception:
        # Fallback if SHAP fails
        explainer = shap.Explainer(model, X_train)

    # Sample 500 test points (to avoid additivity error + improve speed)
    X_test_sample = shap.sample(X_test, 500, random_state=42)

    # Run explainer safely
    shap_values = explainer(X_test_sample, check_additivity=False)

    # === Global Explanations ===
    print("📊 Global SHAP Analysis")
    shap.plots.beeswarm(shap_values, max_display=15)
    plt.show()

    shap.plots.bar(shap_values, max_display=15)
    plt.show()

    # === Local Explanation for Fraud Sample ===
    fraud_indices = np.where(y_test == 1)[0]
    if len(fraud_indices) > 0:
        fraud_index = fraud_indices[0]
        print(f"📌 Explaining fraud sample index: {fraud_index}")
        shap.plots.waterfall(shap_values[0], max_display=10)  # Explain first fraud case
        plt.show()
    else:
        print("⚠️ No fraud samples found in test set for local explanation.")


# === Run SHAP for All Models ===
feature_names = X.columns
for name, model in trained_models.items():
    run_shap_analysis(model, name, X_train, X_test, y_test, feature_names)


Training Logistic Regression...
Training SGD...
Training Ridge...
Training Passive Aggressive...
Training Perceptron...
Training GaussianNB...
Training LinearSVC...
Training Decision Tree...
Training KNN...
Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [15:54:34] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



🔍 Running SHAP analysis for Logistic Regression...


TypeError: LinearExplainer.explain_row() got an unexpected keyword argument 'check_additivity'

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# === Normalize ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

# === GWO Functions ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_gwo(n_wolves, dim, max_iter):
    wolves = np.random.randint(0, 2, size=(n_wolves, dim)).astype(bool)
    fitnesses = np.array([fitness(w) for w in wolves])

    alpha, beta, delta = np.zeros(dim, dtype=bool), np.zeros(dim, dtype=bool), np.zeros(dim, dtype=bool)
    f_alpha, f_beta, f_delta = -np.inf, -np.inf, -np.inf

    print("🐺 Running Grey Wolf Optimization (GWO)...")

    for t in range(max_iter):
        a = 2 - t * (2 / max_iter)

        for i in range(n_wolves):
            fit = fitnesses[i]
            if fit > f_alpha:
                f_alpha, alpha = fit, wolves[i].copy()
            elif fit > f_beta:
                f_beta, beta = fit, wolves[i].copy()
            elif fit > f_delta:
                f_delta, delta = fit, wolves[i].copy()

        for i in range(n_wolves):
            for j in range(dim):
                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha[j] - wolves[i][j])
                X1 = alpha[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta[j] - wolves[i][j])
                X2 = beta[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta[j] - wolves[i][j])
                X3 = delta[j] - A3 * D_delta

                X_avg = (X1 + X2 + X3) / 3
                wolves[i][j] = sigmoid(X_avg) > np.random.rand()

            fitnesses[i] = fitness(wolves[i])

        print(f"Iteration {t+1}/{max_iter} — Best Score: {f_alpha:.4f} — Features: {np.sum(alpha)}")

    return alpha

# === Run GWO ===
n_wolves = 20
max_iter = 20
best_mask = binary_gwo(n_wolves, X_df.shape[1], max_iter)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by Grey Wolf Optimizer:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Model Evaluation ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with GWO-Selected Features:")
print(results_df.to_markdown(index=False))


🐺 Running Grey Wolf Optimization (GWO)...
Iteration 1/20 — Best Score: 0.9768 — Features: 12
Iteration 2/20 — Best Score: 0.9768 — Features: 12
Iteration 3/20 — Best Score: 0.9768 — Features: 12
Iteration 4/20 — Best Score: 0.9768 — Features: 12
Iteration 5/20 — Best Score: 0.9768 — Features: 12
Iteration 6/20 — Best Score: 0.9768 — Features: 12
Iteration 7/20 — Best Score: 0.9768 — Features: 12
Iteration 8/20 — Best Score: 0.9768 — Features: 12
Iteration 9/20 — Best Score: 0.9768 — Features: 12
Iteration 10/20 — Best Score: 0.9768 — Features: 12
Iteration 11/20 — Best Score: 0.9768 — Features: 12
Iteration 12/20 — Best Score: 0.9768 — Features: 12
Iteration 13/20 — Best Score: 0.9769 — Features: 15
Iteration 14/20 — Best Score: 0.9769 — Features: 15
Iteration 15/20 — Best Score: 0.9769 — Features: 15
Iteration 16/20 — Best Score: 0.9769 — Features: 15
Iteration 17/20 — Best Score: 0.9769 — Features: 15
Iteration 18/20 — Best Score: 0.9772 — Features: 16
Iteration 19/20 — Best Score: 0

/tmp/ipython-input-3545322031.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3545322031.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035345 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with GWO-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9826 |     0.1495 | 0.2662 |        0.0252 |    0.9631 |
| NB         |     0.98   |     0.127  | 0.2378 |        0.0194 |    0.9645 |
| SGD        |     0.9697 |     0.0916 | 0.2031 |        0.0316 |    0.961  |
| Ridge      |     0.9954 |     0.3774 | 0.4461 |        0.1301 |    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, brier_score_loss, roc_auc_score

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']
feature_names = X.columns.tolist()

# === Normalize ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

def sigmoid(x): return 1 / (1 + np.exp(-x))

# === Binary PSO ===
def binary_pso(n_particles, dim, max_iter, w=0.7, c1=1.5, c2=1.5):
    position = np.random.rand(n_particles, dim)
    velocity = np.random.randn(n_particles, dim)
    binary_position = position > 0.5
    p_best_position = binary_position.copy()
    p_best_score = np.array([fitness(ind) for ind in binary_position])
    g_best_idx = np.argmax(p_best_score)
    g_best_position = p_best_position[g_best_idx].copy()

    print("🐦 Running Particle Swarm Optimization (PSO)...")
    for t in range(max_iter):
        for i in range(n_particles):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)
            velocity[i] = (
                w * velocity[i]
                + c1 * r1 * (p_best_position[i].astype(float) - binary_position[i].astype(float))
                + c2 * r2 * (g_best_position.astype(float) - binary_position[i].astype(float))
            )
            position[i] = position[i] + velocity[i]
            position[i] = np.clip(position[i], 0, 1)
            binary_position[i] = sigmoid(position[i]) > np.random.rand(dim)

            score = fitness(binary_position[i])
            if score > p_best_score[i]:
                p_best_score[i] = score
                p_best_position[i] = binary_position[i].copy()

        g_best_idx = np.argmax(p_best_score)
        g_best_position = p_best_position[g_best_idx].copy()

        print(f"Iteration {t+1}/{max_iter} — Best Score: {p_best_score[g_best_idx]:.4f} — Features: {np.sum(g_best_position)}")

    return g_best_position

# === Run PSO ===
n_particles = 20
max_iter = 20
best_mask = binary_pso(n_particles, X_df.shape[1], max_iter)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by PSO:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Train-Test Split for Selected Features ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}
if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

model_name_map = {name: name[:4].upper() for name in models}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with PSO-Selected Features:")
print(results_df.to_markdown(index=False))


🐦 Running Particle Swarm Optimization (PSO)...
Iteration 1/20 — Best Score: 0.9791 — Features: 8
Iteration 2/20 — Best Score: 0.9791 — Features: 8
Iteration 3/20 — Best Score: 0.9791 — Features: 8
Iteration 4/20 — Best Score: 0.9791 — Features: 8
Iteration 5/20 — Best Score: 0.9791 — Features: 8
Iteration 6/20 — Best Score: 0.9791 — Features: 8
Iteration 7/20 — Best Score: 0.9791 — Features: 8
Iteration 8/20 — Best Score: 0.9791 — Features: 8
Iteration 9/20 — Best Score: 0.9791 — Features: 8
Iteration 10/20 — Best Score: 0.9791 — Features: 8
Iteration 11/20 — Best Score: 0.9791 — Features: 8
Iteration 12/20 — Best Score: 0.9791 — Features: 8
Iteration 13/20 — Best Score: 0.9791 — Features: 8
Iteration 14/20 — Best Score: 0.9791 — Features: 8
Iteration 15/20 — Best Score: 0.9791 — Features: 8
Iteration 16/20 — Best Score: 0.9791 — Features: 8
Iteration 17/20 — Best Score: 0.9791 — Features: 8
Iteration 18/20 — Best Score: 0.9791 — Features: 8
Iteration 19/20 — Best Score: 0.9791 — Featu

/tmp/ipython-input-608960686.py:44: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))
/tmp/ipython-input-608960686.py:44: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018800 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with PSO-Selected Features:
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LOGI    |     0.9805 |     0.1356 | 0.2522 |        0.0286 |    0.9717 |
| NAIV    |     0.9808 |     0.1321 | 0.2431 |        0.0184 |    0.9689 |
| SGD     |     0.9316 |     0.0437 | 0.1366 |        0.0701 |    0.968  |
| RIDG    |     0.9957 |     0.398  | 0.4641 |        0.1324 |    0.9532 |
| PASS    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, brier_score_loss, roc_auc_score

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']
feature_names = X.columns.tolist()

# === Normalize ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# === Correlation Coefficient Feature Selection ===
correlations = []
for col in X_df.columns:
    corr = np.corrcoef(X_df[col], y)[0, 1]
    correlations.append(abs(corr))

correlation_df = pd.DataFrame({'Feature': X_df.columns, 'Correlation': correlations})
correlation_df = correlation_df.sort_values(by='Correlation', ascending=False)

top_k = 15
selected_features = correlation_df['Feature'].head(top_k).tolist()

print(f"\n✅ Top {top_k} Features Selected by Correlation Coefficient:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Train-Test Split ===
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}
if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

model_name_map = {name: name[:4].upper() for name in models}

def sigmoid(x): return 1 / (1 + np.exp(-x))

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Correlation-Based Features:")
print(results_df.to_markdown(index=False))



✅ Top 15 Features Selected by Correlation Coefficient:
01. V17
02. V14
03. V12
04. V10
05. V16
06. V3
07. V7
08. V11
09. V4
10. V18
11. V1
12. V9
13. V5
14. V2
15. V6


/tmp/ipython-input-3113082272.py:72: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3113082272.py:72: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with Correlation-Based Features:
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LOGI    |     0.9749 |     0.112  | 0.2306 |        0.0263 |    0.9727 |
| NAIV    |     0.986  |     0.1775 | 0.2917 |        0.0136 |    0.958  |
| SGD     |     0.9738 |     0.1064 | 0.2229 |        0.023  |    0.9751 |
| RIDG    |     0.989  |     0.2053 | 0.3088 |        0.1274 |    0.9631 |
| PA

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# === Normalize ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

# === Genetic Algorithm ===
def genetic_algorithm(pop_size, dim, n_generations, crossover_rate=0.8, mutation_rate=0.1):
    population = np.random.randint(0, 2, size=(pop_size, dim)).astype(bool)
    fitness_scores = np.array([fitness(ind) for ind in population])
    print("🧬 Running Genetic Algorithm...")

    for gen in range(n_generations):
        new_population = []

        # Selection (roulette wheel)
        probs = fitness_scores / fitness_scores.sum()
        for _ in range(pop_size // 2):
            parents_idx = np.random.choice(pop_size, size=2, replace=False, p=probs)
            p1, p2 = population[parents_idx]

            # Crossover
            if np.random.rand() < crossover_rate:
                point = np.random.randint(1, dim - 1)
                c1 = np.concatenate([p1[:point], p2[point:]])
                c2 = np.concatenate([p2[:point], p1[point:]])
            else:
                c1, c2 = p1.copy(), p2.copy()

            # Mutation
            for c in [c1, c2]:
                for j in range(dim):
                    if np.random.rand() < mutation_rate:
                        c[j] = not c[j]

            new_population.append(c1)
            new_population.append(c2)

        population = np.array(new_population)
        fitness_scores = np.array([fitness(ind) for ind in population])

        best_idx = np.argmax(fitness_scores)
        print(f"Generation {gen+1}/{n_generations} — Best Score: {fitness_scores[best_idx]:.4f} — Features: {np.sum(population[best_idx])}")

    return population[best_idx]

# === Run GA ===
pop_size = 20
n_generations = 20
best_mask = genetic_algorithm(pop_size, X_df.shape[1], n_generations)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by Genetic Algorithm:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Model Evaluation ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with GA-Selected Features:")
print(results_df.to_markdown(index=False))


🧬 Running Genetic Algorithm...
Generation 1/20 — Best Score: 0.9760 — Features: 12
Generation 2/20 — Best Score: 0.9744 — Features: 14
Generation 3/20 — Best Score: 0.9745 — Features: 15
Generation 4/20 — Best Score: 0.9751 — Features: 14
Generation 5/20 — Best Score: 0.9754 — Features: 13
Generation 6/20 — Best Score: 0.9748 — Features: 15
Generation 7/20 — Best Score: 0.9766 — Features: 15
Generation 8/20 — Best Score: 0.9767 — Features: 16
Generation 9/20 — Best Score: 0.9755 — Features: 19
Generation 10/20 — Best Score: 0.9749 — Features: 16
Generation 11/20 — Best Score: 0.9772 — Features: 12
Generation 12/20 — Best Score: 0.9792 — Features: 9
Generation 13/20 — Best Score: 0.9765 — Features: 15
Generation 14/20 — Best Score: 0.9774 — Features: 14
Generation 15/20 — Best Score: 0.9760 — Features: 14
Generation 16/20 — Best Score: 0.9787 — Features: 13
Generation 17/20 — Best Score: 0.9769 — Features: 12
Generation 18/20 — Best Score: 0.9754 — Features: 13
Generation 19/20 — Best S

/tmp/ipython-input-3081693111.py:130: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3081693111.py:130: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with GA-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9754 |     0.1103 | 0.2249 |        0.0301 |    0.9664 |
| NB         |     0.9861 |     0.1752 | 0.2865 |        0.0131 |    0.9545 |
| SGD        |     0.9109 |     0.0335 | 0.1168 |        0.0775 |    0.9567 |
| Ridge      |     0.9937 |     0.313  | 0.3996 |        0.1314 |    0

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# === Normalize ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# === Train-Test Split ===
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Recursive Feature Elimination (RFE) ===
top_k = 15  # Number of features to select
estimator = LogisticRegression(solver='liblinear', class_weight='balanced')
selector = RFE(estimator=estimator, n_features_to_select=top_k)
selector.fit(X_train_full, y_train)

selected_mask = selector.support_
selected_features = X_df.columns[selected_mask]

print(f"\n✅ Top {top_k} Features Selected by RFE:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Filter the features ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

# === Sigmoid for decision_function if needed ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate All Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Show Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with RFE-Selected Features:")
print(results_df.to_markdown(index=False))



✅ Top 15 Features Selected by RFE:
01. V4
02. V5
03. V6
04. V7
05. V8
06. V10
07. V12
08. V14
09. V16
10. V21
11. V22
12. V23
13. V27
14. V28
15. Amount


/tmp/ipython-input-3250357185.py:88: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3250357185.py:88: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035304 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with RFE-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9747 |     0.1111 | 0.2295 |        0.0256 |    0.9759 |
| NB         |     0.9773 |     0.1139 | 0.2239 |        0.022  |    0.9631 |
| SGD        |     0.9755 |     0.113  | 0.2305 |        0.0296 |    0.9698 |
| Ridge      |     0.9886 |     0.201  | 0.3067 |        0.1258 |    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier, Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, brier_score_loss, roc_auc_score

# try LGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load & preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class']
features = X.columns.tolist()

X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=features)

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, stratify=y, random_state=42)

# === Fitness function ===
def fitness(mask, α=0.01):
    if not np.any(mask): return 0
    sel = X_train.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, sel, y_train, cv=3, scoring='accuracy').mean()
    return acc - α * np.sum(mask)/len(mask)

def sigmoid(x): return 1 / (1 + np.exp(-x))

# === CSO algorithm (binary) ===
def binary_cso(n_cats=30, dim=None, max_iter=20, SMP=5, seeking_rate=0.2, tracing_rate=0.8):
    # initialize random real velocities, positions
    pos = np.random.rand(n_cats, dim)
    vel = np.zeros_like(pos)
    mask = pos > 0.5
    fit = np.array([fitness(m) for m in mask])

    for it in range(max_iter):
        idx_best = np.argmax(fit)
        best_mask = mask[idx_best].copy()

        for i in range(n_cats):
            if np.random.rand() < tracing_rate:
                # tracing mode: move toward global best
                vel[i] = 0.5*vel[i] + np.random.rand(dim)*(best_mask.astype(int) - mask[i].astype(int))
                pos[i] = pos[i] + vel[i]
            else:
                # seeking mode: explore around current
                candidate_positions = []
                for _ in range(SMP):
                    p = mask[i].astype(int).copy()
                    idxs = np.random.choice(dim, size=int(dim*0.1), replace=False)
                    p[idxs] = 1 - p[idxs]
                    candidate_positions.append(p)
                candidate_positions.append(mask[i].astype(int))
                candidate_fits = [fitness(np.array(p, dtype=bool)) for p in candidate_positions]
                mask[i] = np.array(candidate_positions[np.argmax(candidate_fits)], dtype=bool)
                pos[i] = mask[i].astype(float)

        # clip and update
        pos = np.clip(pos, 0, 1)
        mask = pos > 0.5
        fit = np.array([fitness(m) for m in mask])

        print(f"Iter {it+1}/{max_iter} — Best fitness: {fit[idx_best]:.4f}, Features: {mask[idx_best].sum()}")

    return mask[idx_best]

# === Run CSO ===
best_mask = binary_cso(n_cats=30, dim=X_df.shape[1], max_iter=20)
selected = X_df.columns[best_mask]

print("\n✅ Selected Features by Cat Swarm Optimization:")
for idx,f in enumerate(selected, 1):
    print(f"{idx:02d}. {f}")

# === Evaluate classifiers ===
X_tr = X_train[selected]
X_te = X_test[selected]

models = {
    "Logistic Regression": LogisticRegression(solver='liblinear', class_weight='balanced'),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight='balanced', random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight='balanced'),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight='balanced', random_state=42),
    "Perceptron": Perceptron(class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}
if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

model_map = {nm: nm[:4].upper() for nm in models}

results = []
for name, mdl in models.items():
    mdl.fit(X_tr, y_train)
    y_pred = mdl.predict(X_te)
    if hasattr(mdl, "predict_proba"):
        y_prob = mdl.predict_proba(X_te)[:,1]
    elif hasattr(mdl, "decision_function"):
        y_prob = sigmoid(mdl.decision_function(X_te))
    else:
        y_prob = None

    results.append({
        "Model": model_map[name],
        "Accuracy": round(accuracy_score(y_test,y_pred),4),
        "F1 Score": round(f1_score(y_test,y_pred),4),
        "MCC": round(matthews_corrcoef(y_test,y_pred),4),
        "Brier Score": round(brier_score_loss(y_test,y_prob),4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test,y_prob),4) if y_prob is not None else np.nan
    })

res_df = pd.DataFrame(results)
print("\n📊 Performance with CSO-Selected Features:")
print(res_df.to_markdown(index=False))


Iter 1/20 — Best fitness: 0.9752, Features: 17
Iter 2/20 — Best fitness: 0.9780, Features: 13
Iter 3/20 — Best fitness: 0.9783, Features: 12
Iter 4/20 — Best fitness: 0.9815, Features: 10
Iter 5/20 — Best fitness: 0.9815, Features: 10
Iter 6/20 — Best fitness: 0.9815, Features: 10
Iter 7/20 — Best fitness: 0.9815, Features: 10
Iter 8/20 — Best fitness: 0.9817, Features: 11
Iter 9/20 — Best fitness: 0.9817, Features: 11
Iter 10/20 — Best fitness: 0.9823, Features: 8
Iter 11/20 — Best fitness: 0.9824, Features: 9
Iter 12/20 — Best fitness: 0.9824, Features: 9
Iter 13/20 — Best fitness: 0.9825, Features: 8
Iter 14/20 — Best fitness: 0.9827, Features: 7
Iter 15/20 — Best fitness: 0.9827, Features: 7
Iter 16/20 — Best fitness: 0.9827, Features: 7
Iter 17/20 — Best fitness: 0.9827, Features: 7
Iter 18/20 — Best fitness: 0.9827, Features: 7
Iter 19/20 — Best fitness: 0.9827, Features: 7
Iter 20/20 — Best fitness: 0.9827, Features: 7

✅ Selected Features by Cat Swarm Optimization:
01. V9
02. V

/tmp/ipython-input-3398248382.py:37: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3398248382.py:37: RuntimeWarning: overflow encountered in exp
  def sigmoid(x): return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018814 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Performance with CSO-Selected Features:
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LOGI    |     0.9837 |     0.1576 | 0.2742 |        0.0261 |    0.9684 |
| NAIV    |     0.9882 |     0.2017 | 0.3121 |        0.0112 |    0.9636 |
| SGD     |     0.9833 |     0.1548 | 0.2715 |        0.0448 |    0.966  |
| RIDG    |     0.9946 |     0.3462 | 0.4238 |        0.1321 |    0.9542 |
| PASS    |     

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# === Chi-Square requires non-negative data ===
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# === Train-Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# === Feature Selection using Chi-Square ===
top_k = 15
selector = SelectKBest(score_func=chi2, k=top_k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

selected_mask = selector.get_support()
selected_features = X_df.columns[selected_mask]
print(f"\n✅ Top {top_k} Features Selected by Chi-Square Test:")
for i, fname in enumerate(selected_features, 1):
    print(f"{i:02d}. {fname}")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate All Models ===
results = []
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_sel)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test_sel))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Show Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Chi-Square Selected Features:")
print(results_df.to_markdown(index=False))



✅ Top 15 Features Selected by Chi-Square Test:
01. Time
02. V1
03. V2
04. V3
05. V4
06. V7
07. V9
08. V10
09. V11
10. V12
11. V14
12. V16
13. V17
14. V18
15. V19


/tmp/ipython-input-571729525.py:82: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-571729525.py:82: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024820 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



📊 Model Performance with Chi-Square Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9755 |     0.1141 | 0.233  |        0.0263 |    0.9695 |
| NB         |     0.987  |     0.1884 | 0.3018 |        0.0127 |    0.9627 |
| SGD        |     0.9492 |     0.0586 | 0.1616 |        0.0443 |    0.9657 |
| Ridge      |     0.9891 |     0.2087 | 0.3134 |        0.1275 |    0.9627 |
| PA         |     0.9941 |     0.3379 | 0.4269 |        0.0059 |    0.9609 |
| Perceptron |     0.9941 |     0.3373 | 0.4264 |        0.0059 |    0.961  |
| DT         |     0.9991 |     0.7374 | 0.737  |        0.0009 |    0.8722 |
| ET         |     0.9996 |     0.8681 | 0.8705 |        0.0004 |    0.9478 |
| ADA        |     0.999  |     0.7035 | 0.7031 |        0.0857 |    0.9696 |
| LGBM       |     0.9991 |     0.7727 | 0.7769 |        0.0008 |    0.9742 |


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Normalize (recommended for MI)
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=X.columns)

# === Split Data ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# === Feature Selection with Mutual Information ===
top_k = 15
selector = SelectKBest(score_func=mutual_info_classif, k=top_k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

selected_mask = selector.get_support()
selected_features = X.columns[selected_mask]
print(f"\n✅ Top {top_k} Features Selected by Information Gain (Mutual Information):")
for i, fname in enumerate(selected_features, 1):
    print(f"{i:02d}. {fname}")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_sel)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test_sel))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Output Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Information Gain (Top 15 Features):")
print(results_df.to_markdown(index=False))



✅ Top 15 Features Selected by Information Gain (Mutual Information):
01. V2
02. V3
03. V4
04. V5
05. V7
06. V9
07. V10
08. V11
09. V12
10. V14
11. V16
12. V17
13. V18
14. V21
15. V27


/tmp/ipython-input-2221229630.py:81: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-2221229630.py:81: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030252 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



📊 Model Performance with Information Gain (Top 15 Features):
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9743 |     0.1096 | 0.2278 |        0.0261 |    0.9725 |
| NB         |     0.9829 |     0.15   | 0.2653 |        0.0167 |    0.9616 |
| SGD        |     0.9835 |     0.1578 | 0.2759 |        0.0175 |    0.9705 |
| Ridge      |     0.9922 |     0.266  | 0.36   |        0.1277 |    0.9598 |
| PA         |     0.994  |     0.3379 | 0.4288 |        0.0059 |    0.9671 |
| Perceptron |     0.9939 |     0.3346 | 0.4262 |        0.006  |    0.9672 |
| DT         |     0.999  |     0.6943 | 0.6939 |        0.001  |    0.8416 |
| ET         |     0.9996 |     0.8681 | 0.8705 |        0.0004 |    0.953  |
| ADA        |     0.9991 |     0.7196 | 0.7196 |        0.0881 |    0.9693 |
| LGBM       |     0.9991 |     0.7742 | 0.7774 |        0.0008 |    0.9784 |


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load and Preprocess Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Normalize features
X_scaled = MinMaxScaler().fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# === Correlation Coefficient Feature Selection ===
# Compute Pearson correlation between each feature and the target
correlations = X_scaled_df.apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))
top_k = 15
top_features = correlations.sort_values(ascending=False).head(top_k).index.tolist()

print(f"✅ Selected Top {len(top_features)} features using Pearson Correlation.")
X_selected = X_scaled_df[top_features]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, stratify=y, random_state=42
)

# === Define Classifiers ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# === Short Names for Display ===
model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

# === Helper: Sigmoid function ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate All Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        decision_scores = model.decision_function(X_test)
        y_prob = sigmoid(decision_scores)
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Output Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Pearson Correlation (Top 15 Features):")
print(results_df.to_markdown(index=False))


✅ Selected Top 15 features using Pearson Correlation.


/tmp/ipython-input-2949844582.py:80: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-2949844582.py:80: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043327 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with Pearson Correlation (Top 15 Features):
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9749 |     0.112  | 0.2306 |        0.0263 |    0.9727 |
| NB         |     0.986  |     0.1775 | 0.2917 |        0.0136 |    0.958  |
| SGD        |     0.9738 |     0.1064 | 0.2229 |        0.023  |    0.9751 |
| Ridge      |     0.989  |     0.2053 | 0.3088 |    

In [ ]:
pip install pyswarm


  Preparing metadata (setup.py) ... done
  Created wheel for pyswarm: filename=pyswarm-0.6-py3-none-any.whl size=4463 sha256=af74346128f79301c157a575cc957ef2c14f6b5f5771ead182b03dfaeb716d74
  Stored in directory: /root/.cache/pip/wheels/bb/4f/ec/8970b83323e16aa95034da175454843947376614d6d5e9627f
Successfully built pyswarm


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from pyswarm import pso

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=X.columns)

# === Split Early for Fitness Function ===
X_train_full, X_test_full, y_train, y_test = train_test_split(X_df, y, test_size=0.2, stratify=y, random_state=42)

# === PSO Fitness Function ===
def fitness_function(particle):
    particle = np.array(particle)
    mask = particle > 0.5  # binary mask

    if not any(mask):
        return 1.0  # worst score

    X_selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')

    # Use negative mean accuracy
    score = cross_val_score(model, X_selected, y_train, scoring='accuracy', cv=3).mean()

    # Fitness: (1 - accuracy) + penalty for #features
    return 1 - score + (np.sum(mask) / len(mask)) * 0.01  # small penalty for more features

# === Run PSO ===
num_features = X_df.shape[1]
lb = [0] * num_features
ub = [1] * num_features

print("🚀 Running PSO for Feature Selection...")
best_position, _ = pso(fitness_function, lb, ub, swarmsize=30, maxiter=20)

# === Select Features ===
selected_mask = np.array(best_position) > 0.5
selected_features = X_df.columns[selected_mask]
print(f"✅ PSO selected {len(selected_features)} features: {list(selected_features)}")

# Subset data
X_selected_train = X_train_full[selected_features]
X_selected_test = X_test_full[selected_features]

# === Define Classifiers ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# Short names
model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

# Sigmoid helper
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_selected_train, y_train)
    y_pred = model.predict(X_selected_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_selected_test)[:, 1]
    elif hasattr(model, "decision_function"):
        decision_scores = model.decision_function(X_selected_test)
        y_prob = sigmoid(decision_scores)
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Output Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with PSO-selected Features:")
print(results_df.to_markdown(index=False))


In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load and Preprocess Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=X.columns)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function for SA ===
def evaluate(mask, alpha=0.01):
    if not np.any(mask):
        return 1.0  # penalize if no features selected
    X_selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    score = cross_val_score(model, X_selected, y_train, scoring='accuracy', cv=3).mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return 1 - score + penalty

# === Simulated Annealing ===
def simulated_annealing(n_features, max_iter=1000, initial_temp=100, cooling_rate=0.95):
    current_mask = np.random.choice([True, False], size=n_features)
    current_score = evaluate(current_mask)
    best_mask = current_mask.copy()
    best_score = current_score
    temp = initial_temp

    for i in range(max_iter):
        # Generate neighbor
        neighbor = current_mask.copy()
        flip_index = random.randint(0, n_features - 1)
        neighbor[flip_index] = not neighbor[flip_index]

        neighbor_score = evaluate(neighbor)
        delta = neighbor_score - current_score

        # Accept neighbor?
        if delta < 0 or np.exp(-delta / temp) > np.random.rand():
            current_mask = neighbor
            current_score = neighbor_score

            if current_score < best_score:
                best_mask = current_mask.copy()
                best_score = current_score

        temp *= cooling_rate  # cool down

    return best_mask

# === Run SA ===
print("🔥 Running Simulated Annealing for Feature Selection...")
n_features = X_df.shape[1]
best_mask = simulated_annealing(n_features=n_features, max_iter=200, initial_temp=50, cooling_rate=0.9)
selected_features = X_df.columns[best_mask]
print(f"✅ SA selected {len(selected_features)} features: {list(selected_features)}")

# === Final Data Preparation ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

# === Define Classifiers ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# Short names
model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

# Sigmoid helper
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        decision_scores = model.decision_function(X_test)
        y_prob = sigmoid(decision_scores)
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Output Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Simulated Annealing (Selected Features):")
print(results_df.to_markdown(index=False))


🔥 Running Simulated Annealing for Feature Selection...
✅ SA selected 5 features: ['V10', 'V12', 'V14', 'V16', 'V21']


/tmp/ipython-input-3699101578.py:119: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3699101578.py:119: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019015 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with Simulated Annealing (Selected Features):
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9824 |     0.1478 | 0.2646 |        0.0265 |    0.9683 |
| NB         |     0.9884 |     0.2051 | 0.3152 |        0.011  |    0.9626 |
| SGD        |     0.9856 |     0.1751 | 0.2909 |        0.0474 |    0.9665 |
| Ridge      |     0.9961 |     0.4267 | 0.4902 |   

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load and Preprocess Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# Scale
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# Train/Test Split
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask):
        return 0
    X_selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    score = cross_val_score(model, X_selected, y_train, scoring='accuracy', cv=3).mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return score - penalty  # higher is better

# === ACO Parameters ===
num_features = X_df.shape[1]
num_ants = 20
num_iterations = 20
evaporation_rate = 0.1
pheromone = np.ones(num_features)
pheromone_min = 0.1
pheromone_max = 5.0

best_solution = None
best_score = -np.inf

print("🐜 Running Ant Colony Optimization for Feature Selection...")

for iteration in range(num_iterations):
    all_solutions = []
    all_scores = []

    for _ in range(num_ants):
        prob = pheromone / pheromone.sum()
        solution = np.random.rand(num_features) < prob
        score = fitness(solution)

        all_solutions.append(solution)
        all_scores.append(score)

        if score > best_score:
            best_score = score
            best_solution = solution.copy()

    # Evaporation and Update
    pheromone *= (1 - evaporation_rate)
    for solution, score in zip(all_solutions, all_scores):
        pheromone += score * solution

    pheromone = np.clip(pheromone, pheromone_min, pheromone_max)

    print(f"Iteration {iteration + 1}/{num_iterations} — Best Score: {best_score:.4f} — Features Selected: {int(np.sum(best_solution))}")

# === Final Selected Features ===
selected_feature_names = X_df.columns[best_solution]
print("\n✅ Final Selected Features by ACO:")
for i, fname in enumerate(selected_feature_names, 1):
    print(f"{i:02d}. {fname}")

# === Show All Features Ranked by Pheromone ===
sorted_indices = np.argsort(-pheromone)  # descending
print("\n📈 Feature Ranking by Final Pheromone Strength:")
for i in sorted_indices[:15]:
    print(f"{feature_names[i]:<10}  --> pheromone = {pheromone[i]:.4f}")

# Prepare Final Data
X_train = X_train_full[selected_feature_names]
X_test = X_test_full[selected_feature_names]

# === Classifiers ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# === Short names ===
model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

# === Sigmoid for decision_function fallback ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate All Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Show Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with ACO-Selected Features:")
print(results_df.to_markdown(index=False))


🐜 Running Ant Colony Optimization for Feature Selection...
Iteration 1/20 — Best Score: 0.9695 — Features Selected: 1
Iteration 2/20 — Best Score: 0.9724 — Features Selected: 1
Iteration 3/20 — Best Score: 0.9802 — Features Selected: 3
Iteration 4/20 — Best Score: 0.9802 — Features Selected: 3
Iteration 5/20 — Best Score: 0.9802 — Features Selected: 3
Iteration 6/20 — Best Score: 0.9802 — Features Selected: 3
Iteration 7/20 — Best Score: 0.9940 — Features Selected: 2
Iteration 8/20 — Best Score: 0.9940 — Features Selected: 2
Iteration 9/20 — Best Score: 0.9940 — Features Selected: 2
Iteration 10/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 11/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 12/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 13/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 14/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 15/20 — Best Score: 0.9981 — Features Selected: 1
Iteration 16/20 — Best Score: 0.9981 — Features Se

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# Normalize (required for Fisher)
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# === Fisher Score Calculation ===
def fisher_score(X, y):
    scores = []
    for feature in X.columns:
        x = X[feature]
        x0 = x[y == 0]
        x1 = x[y == 1]
        μ0 = np.mean(x0)
        μ1 = np.mean(x1)
        σ0 = np.var(x0)
        σ1 = np.var(x1)
        μ = np.mean(x)

        numerator = (μ0 - μ) ** 2 + (μ1 - μ) ** 2
        denominator = σ0 + σ1 + 1e-10  # avoid divide by 0
        score = numerator / denominator
        scores.append(score)
    return np.array(scores)

# === Select Top k Features ===
print("🔍 Computing Fisher's Score...")
scores = fisher_score(X_df, y)
top_k = 15
top_indices = np.argsort(scores)[-top_k:][::-1]
selected_features = X_df.columns[top_indices]
print(f"\n✅ Top {top_k} Features Selected by Fisher’s Score:")
for i, fname in enumerate(selected_features, 1):
    print(f"{i:02d}. {fname}")

# === Subset Data ===
X_selected = X_df[selected_features]
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, stratify=y, random_state=42
)

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Show Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with Fisher's Score (Top 15 Features):")
print(results_df.to_markdown(index=False))


🔍 Computing Fisher's Score...

✅ Top 15 Features Selected by Fisher’s Score:
01. V14
02. V4
03. V11
04. V12
05. V10
06. V16
07. V3
08. V17
09. V9
10. V2
11. V7
12. V18
13. V1
14. V6
15. V5


/tmp/ipython-input-1738484438.py:99: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-1738484438.py:99: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035256 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with Fisher's Score (Top 15 Features):
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9749 |     0.112  | 0.2306 |        0.0263 |    0.9727 |
| NB         |     0.986  |     0.1775 | 0.2917 |        0.0136 |    0.958  |
| SGD        |     0.9738 |     0.1064 | 0.2229 |        0.023  |    0.9751 |
| Ridge      |     0.989  |     0.2053 | 0.3088 |        0

In [ ]:
pip install scikit-learn pandas numpy


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef, brier_score_loss, roc_auc_score
)

# === Load Data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# === Split & Standardize ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Information Gain Feature Selection ===
print("Calculating information gain...")
info_gain = mutual_info_classif(X_train, y_train, discrete_features=False, random_state=42)
top_k = 15  # number of top features to select
selected_indices = np.argsort(info_gain)[-top_k:]  # top-k indices
X_train_sel = X_train[:, selected_indices]
X_test_sel = X_test[:, selected_indices]
print(f"Selected Top {top_k} features using Information Gain.")

# === Define Classifiers ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
        n_jobs=-1
    )
}

# === Short Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "Random Forest": "RF",
    "XGBoost": "XGB"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Display Results ===
results_df = pd.DataFrame(results)
print("\n✅ Model Performance with Information Gain (Top 15 Features):")
print(results_df.to_markdown(index=False))


📘 Running Teaching-Learning-Based Optimization (TLBO)...
Generation 1/20 — Best Score: 0.9862 — Features: 6
Generation 2/20 — Best Score: 0.9862 — Features: 6
Generation 3/20 — Best Score: 0.9862 — Features: 6
Generation 4/20 — Best Score: 0.9862 — Features: 6
Generation 5/20 — Best Score: 0.9862 — Features: 6
Generation 6/20 — Best Score: 0.9862 — Features: 6
Generation 7/20 — Best Score: 0.9862 — Features: 6
Generation 8/20 — Best Score: 0.9862 — Features: 6
Generation 9/20 — Best Score: 0.9862 — Features: 6
Generation 10/20 — Best Score: 0.9862 — Features: 6
Generation 11/20 — Best Score: 0.9862 — Features: 6
Generation 12/20 — Best Score: 0.9862 — Features: 6
Generation 13/20 — Best Score: 0.9862 — Features: 6
Generation 14/20 — Best Score: 0.9862 — Features: 6
Generation 15/20 — Best Score: 0.9862 — Features: 6
Generation 16/20 — Best Score: 0.9862 — Features: 6
Generation 17/20 — Best Score: 0.9862 — Features: 6
Generation 18/20 — Best Score: 0.9862 — Features: 6
Generation 19/20

/tmp/ipython-input-2764284812.py:142: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025211 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with TLBO-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9876 |     0.1812 | 0.2826 |        0.0831 |    0.8569 |
| NB         |     0.9911 |     0.2266 | 0.315  |        0.0082 |    0.9495 |
| SGD        |     0.9985 |     0.6207 | 0.6276 |        0.0583 |    0.852  |
| Ridge      |     0.9941 |     0.2905 | 0.3571 |        0.1734 |    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# Normalize for consistency
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

# === Whale Optimization Algorithm (Binary WOA) ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_woa(n_whales, n_features, max_iter):
    # Initialize whales randomly (binary masks)
    whales = np.random.randint(0, 2, size=(n_whales, n_features)).astype(bool)
    fitnesses = np.array([fitness(w) for w in whales])

    leader_idx = np.argmax(fitnesses)
    leader = whales[leader_idx].copy()
    leader_score = fitnesses[leader_idx]

    print("🐋 Running Whale Optimization Algorithm (WOA)...")

    for t in range(max_iter):
        a = 2 - t * (2 / max_iter)  # linearly decreasing from 2 to 0

        for i in range(n_whales):
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2 * a * r1 - a
            C = 2 * r2
            p = np.random.rand()

            if p < 0.5:
                if np.abs(A) < 1:
                    D = np.abs(C * leader.astype(int) - whales[i].astype(int))
                    new_pos = leader.astype(int) - A * D
                else:
                    rand_whale = whales[np.random.randint(0, n_whales)].astype(int)
                    D = np.abs(C * rand_whale - whales[i].astype(int))
                    new_pos = rand_whale - A * D
            else:
                distance_to_leader = np.abs(leader.astype(int) - whales[i].astype(int))
                b = 1
                l = np.random.uniform(-1, 1)
                new_pos = distance_to_leader * np.exp(b * l) * np.cos(2 * np.pi * l) + leader.astype(int)

            # Convert to binary using sigmoid
            s = sigmoid(new_pos)
            new_binary = np.random.rand(n_features) < s
            new_fitness = fitness(new_binary)

            if new_fitness > fitnesses[i]:
                whales[i] = new_binary
                fitnesses[i] = new_fitness

                if new_fitness > leader_score:
                    leader = new_binary.copy()
                    leader_score = new_fitness

        print(f"Iteration {t+1}/{max_iter} — Best Score: {leader_score:.4f} — Features: {np.sum(leader)}")

    return leader

# === Run WOA ===
n_whales = 20
n_iter = 20
best_mask = binary_woa(n_whales, X_df.shape[1], n_iter)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by Whale Optimization:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Train Classifiers ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

# === Display Results ===
results_df = pd.DataFrame(results)
print("\n📊 Model Performance with WOA-Selected Features:")
print(results_df.to_markdown(index=False))


🐋 Running Whale Optimization Algorithm (WOA)...
Iteration 1/20 — Best Score: 0.9804 — Features: 11
Iteration 2/20 — Best Score: 0.9804 — Features: 11
Iteration 3/20 — Best Score: 0.9804 — Features: 11
Iteration 4/20 — Best Score: 0.9804 — Features: 11
Iteration 5/20 — Best Score: 0.9804 — Features: 11
Iteration 6/20 — Best Score: 0.9804 — Features: 11
Iteration 7/20 — Best Score: 0.9804 — Features: 11
Iteration 8/20 — Best Score: 0.9804 — Features: 11
Iteration 9/20 — Best Score: 0.9804 — Features: 11
Iteration 10/20 — Best Score: 0.9804 — Features: 11
Iteration 11/20 — Best Score: 0.9804 — Features: 11
Iteration 12/20 — Best Score: 0.9804 — Features: 11
Iteration 13/20 — Best Score: 0.9804 — Features: 11
Iteration 14/20 — Best Score: 0.9804 — Features: 11
Iteration 15/20 — Best Score: 0.9804 — Features: 11
Iteration 16/20 — Best Score: 0.9804 — Features: 11
Iteration 17/20 — Best Score: 0.9804 — Features: 11
Iteration 18/20 — Best Score: 0.9804 — Features: 11
Iteration 19/20 — Best Sc

/tmp/ipython-input-3254807781.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-3254807781.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040791 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with WOA-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9834 |     0.1554 | 0.272  |        0.0268 |    0.9681 |
| NB         |     0.9844 |     0.1588 | 0.271  |        0.015  |    0.9652 |
| SGD        |     0.9418 |     0.0504 | 0.1471 |        0.0683 |    0.9575 |
| Ridge      |     0.9944 |     0.3388 | 0.42   |        0.1319 |    

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# Normalize
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

# === EO Parameters ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_equilibrium_optimizer(pop_size, dim, max_iter):
    # Initialize population
    population = np.random.rand(pop_size, dim)
    binary_pop = population > 0.5
    fitnesses = np.array([fitness(ind) for ind in binary_pop])
    equilibrium_pool = population[np.argsort(fitnesses)[-4:]]  # Top 4

    best_fitness = np.max(fitnesses)
    best_solution = binary_pop[np.argmax(fitnesses)]

    print("⚖️ Running Equilibrium Optimizer (EO)...")

    t = 1
    a1, a2 = 2, 1
    GP = 0.5  # Generation probability

    for it in range(max_iter):
        T = (1 - (t / max_iter)) ** (a2 * t / max_iter)

        for i in range(pop_size):
            r1, r2 = np.random.rand(), np.random.rand()
            GCP = 0.5 * r1 if r2 >= GP else 0
            r3, r4 = np.random.rand(), np.random.rand()

            eq_idx = np.random.randint(0, 4)
            C_eq = equilibrium_pool[eq_idx]

            F = a1 * np.sign(r3 - 0.5) * (np.exp(-r4 * T) - 1)
            G0 = GCP * (C_eq - r3 * population[i])
            G = G0 * F

            X_new = C_eq + (population[i] - C_eq) * F + G * T
            X_new = np.clip(X_new, 0, 1)
            B_new = sigmoid(X_new) > np.random.rand(dim)

            f_new = fitness(B_new)
            if f_new > fitnesses[i]:
                population[i] = X_new
                fitnesses[i] = f_new

        # Update equilibrium pool
        equilibrium_pool = population[np.argsort(fitnesses)[-4:]]
        if np.max(fitnesses) > best_fitness:
            best_fitness = np.max(fitnesses)
            best_solution = (population[np.argmax(fitnesses)] > 0.5)

        print(f"Iteration {it+1}/{max_iter} — Best Score: {best_fitness:.4f} — Features: {np.sum(best_solution)}")
        t += 1

    return best_solution

# === Run EO ===
pop_size = 20
max_iter = 20
best_mask = binary_equilibrium_optimizer(pop_size, X_df.shape[1], max_iter)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by Equilibrium Optimizer:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Model Evaluation ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with EO-Selected Features:")
print(results_df.to_markdown(index=False))


⚖️ Running Equilibrium Optimizer (EO)...
Iteration 1/20 — Best Score: 0.9764 — Features: 14
Iteration 2/20 — Best Score: 0.9764 — Features: 14
Iteration 3/20 — Best Score: 0.9764 — Features: 14
Iteration 4/20 — Best Score: 0.9764 — Features: 14
Iteration 5/20 — Best Score: 0.9764 — Features: 14
Iteration 6/20 — Best Score: 0.9764 — Features: 14
Iteration 7/20 — Best Score: 0.9783 — Features: 16
Iteration 8/20 — Best Score: 0.9783 — Features: 16
Iteration 9/20 — Best Score: 0.9783 — Features: 16
Iteration 10/20 — Best Score: 0.9783 — Features: 16
Iteration 11/20 — Best Score: 0.9783 — Features: 16
Iteration 12/20 — Best Score: 0.9783 — Features: 16
Iteration 13/20 — Best Score: 0.9783 — Features: 16
Iteration 14/20 — Best Score: 0.9783 — Features: 16
Iteration 15/20 — Best Score: 0.9783 — Features: 16
Iteration 16/20 — Best Score: 0.9783 — Features: 16
Iteration 17/20 — Best Score: 0.9783 — Features: 16
Iteration 18/20 — Best Score: 0.9783 — Features: 16
Iteration 19/20 — Best Score: 0.

/tmp/ipython-input-1371572629.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipython-input-1371572629.py:49: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


[LightGBM] [Info] Number of positive: 394, number of negative: 227451
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011371 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 227845, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

📊 Model Performance with EO-Selected Features:
| Model      |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:-----------|-----------:|-----------:|-------:|--------------:|----------:|
| LR         |     0.9699 |     0.0922 | 0.2038 |        0.0373 |    0.9606 |
| NB         |     0.9771 |     0.1107 | 0.2178 |        0.0222 |    0.9579 |
| SGD        |     0.9775 |     0.1146 | 0.2247 |        0.0205 |    0.9474 |
| Ridg

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# Optional: LightGBM
try:
    from lightgbm import LGBMClassifier
    use_lgbm = True
except ImportError:
    use_lgbm = False

# === Load Dataset ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]
feature_names = X.columns.tolist()

# Normalize
X_scaled = MinMaxScaler().fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=feature_names)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# === Fitness Function ===
def fitness(mask, alpha=0.01):
    if not np.any(mask): return 0
    selected = X_train_full.loc[:, mask]
    model = LogisticRegression(solver='liblinear', class_weight='balanced')
    acc = cross_val_score(model, selected, y_train, cv=3, scoring='accuracy').mean()
    penalty = alpha * np.sum(mask) / len(mask)
    return acc - penalty

# === EO Parameters ===
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_equilibrium_optimizer(pop_size, dim, max_iter):
    # Initialize population
    population = np.random.rand(pop_size, dim)
    binary_pop = population > 0.5
    fitnesses = np.array([fitness(ind) for ind in binary_pop])
    equilibrium_pool = population[np.argsort(fitnesses)[-4:]]  # Top 4

    best_fitness = np.max(fitnesses)
    best_solution = binary_pop[np.argmax(fitnesses)]

    print("⚖️ Running Equilibrium Optimizer (EO)...")

    t = 1
    a1, a2 = 2, 1
    GP = 0.5  # Generation probability

    for it in range(max_iter):
        T = (1 - (t / max_iter)) ** (a2 * t / max_iter)

        for i in range(pop_size):
            r1, r2 = np.random.rand(), np.random.rand()
            GCP = 0.5 * r1 if r2 >= GP else 0
            r3, r4 = np.random.rand(), np.random.rand()

            eq_idx = np.random.randint(0, 4)
            C_eq = equilibrium_pool[eq_idx]

            F = a1 * np.sign(r3 - 0.5) * (np.exp(-r4 * T) - 1)
            G0 = GCP * (C_eq - r3 * population[i])
            G = G0 * F

            X_new = C_eq + (population[i] - C_eq) * F + G * T
            X_new = np.clip(X_new, 0, 1)
            B_new = sigmoid(X_new) > np.random.rand(dim)

            f_new = fitness(B_new)
            if f_new > fitnesses[i]:
                population[i] = X_new
                fitnesses[i] = f_new

        # Update equilibrium pool
        equilibrium_pool = population[np.argsort(fitnesses)[-4:]]
        if np.max(fitnesses) > best_fitness:
            best_fitness = np.max(fitnesses)
            best_solution = (population[np.argmax(fitnesses)] > 0.5)

        print(f"Iteration {it+1}/{max_iter} — Best Score: {best_fitness:.4f} — Features: {np.sum(best_solution)}")
        t += 1

    return best_solution

# === Run EO ===
pop_size = 20
max_iter = 20
best_mask = binary_equilibrium_optimizer(pop_size, X_df.shape[1], max_iter)
selected_features = X_df.columns[best_mask]

print("\n✅ Selected Features by Equilibrium Optimizer:")
for i, f in enumerate(selected_features, 1):
    print(f"{i:02d}. {f}")

# === Model Evaluation ===
X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "Naive Bayes": GaussianNB(),
    "SGD": SGDClassifier(class_weight="balanced", random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
    "Passive Aggressive": PassiveAggressiveClassifier(class_weight="balanced", random_state=42),
    "Perceptron": Perceptron(class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

if use_lgbm:
    models["LightGBM"] = LGBMClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

model_name_map = {
    "Logistic Regression": "LR",
    "Naive Bayes": "NB",
    "SGD": "SGD",
    "Ridge Classifier": "Ridge",
    "Passive Aggressive": "PA",
    "Perceptron": "Perceptron",
    "Decision Tree": "DT",
    "Extra Trees": "ET",
    "AdaBoost": "ADA",
    "LightGBM": "LGBM"
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = sigmoid(model.decision_function(X_test))
    else:
        y_prob = None

    results.append({
        "Model": model_name_map.get(name, name),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4) if y_prob is not None else np.nan,
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4) if y_prob is not None else np.nan
    })

results_df = pd.DataFrame(results)
print("\n📊 Model Performance with EO-Selected Features:")
print(results_df.to_markdown(index=False))


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# === Load and preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Chi-Square requires non-negative input, so scale accordingly
X = MinMaxScaler().fit_transform(X)  # Chi2 doesn't support negative values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Chi-Square Feature Selection ===
top_k = 15  # You can tune this value
selector = SelectKBest(score_func=chi2, k=top_k)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)
selected_features = selector.get_support()

print(f"Selected {np.sum(selected_features)} features using Chi-Square.")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_chi2 = pd.DataFrame(results)
print("\n📊 Model Performance (Chi-Square selected features):")
print(results_df_chi2.to_markdown(index=False))


Selected 15 features using Chi-Square.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:56:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



📊 Model Performance (Chi-Square selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9755 |     0.1141 | 0.233  |        0.0263 |    0.9695 |
| KNN     |     0.9995 |     0.8556 | 0.8587 |        0.0004 |    0.9183 |
| XGB     |     0.9994 |     0.8187 | 0.8184 |        0.0006 |    0.9699 |
| RF      |     0.9995 |     0.8492 | 0.8528 |        0.0004 |    0.9529 |


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# === Load and preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# Scale features for model training (not for correlation calculation)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === Compute Pearson Correlation Coefficients ===
correlations = X.corrwith(y).abs()  # Absolute values of correlation
top_k = 15  # Number of top correlated features to select
top_features = correlations.sort_values(ascending=False).head(top_k).index.tolist()

# Filter dataset by top correlated features
X = X[top_features]

# === Split and scale selected features ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Selected {len(top_features)} features based on correlation coefficient.")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_corr = pd.DataFrame(results)
print("\n📊 Model Performance (Correlation Coefficient selected features):")
print(results_df_corr.to_markdown(index=False))


Selected 15 features based on correlation coefficient.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [05:01:56] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



📊 Model Performance (Correlation Coefficient selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9732 |     0.1056 | 0.2233 |        0.027  |    0.9699 |
| KNN     |     0.9995 |     0.8602 | 0.8612 |        0.0004 |    0.9285 |
| XGB     |     0.9994 |     0.8218 | 0.8218 |        0.0006 |    0.9828 |
| RF      |     0.9995 |     0.8475 | 0.8522 |        0.0004 |    0.9582 |


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# === Load data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# === Fisher Score Calculation ===
def fisher_score(X, y):
    scores = []
    for col in X.columns:
        values_0 = X.loc[y == 0, col]
        values_1 = X.loc[y == 1, col]
        mean_0, mean_1 = np.mean(values_0), np.mean(values_1)
        var_0, var_1 = np.var(values_0), np.var(values_1)
        score = (mean_0 - mean_1)**2 / (var_0 + var_1 + 1e-8)  # Avoid div by zero
        scores.append(score)
    return np.array(scores)

# === Select Top-k Features ===
fisher_scores = fisher_score(X, y)
top_k = 15
top_indices = np.argsort(fisher_scores)[-top_k:][::-1]
top_features = X.columns[top_indices]
X = X[top_features]

# === Train/Test Split & Scale ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Selected {len(top_features)} features based on Fisher's Score.")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Results ===
results_df_fisher = pd.DataFrame(results)
print("\n📊 Model Performance (Fisher's Score selected features):")
print(results_df_fisher.to_markdown(index=False))


Selected 15 features based on Fisher's Score.


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [05:05:07] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



📊 Model Performance (Fisher's Score selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9732 |     0.1056 | 0.2233 |        0.027  |    0.9699 |
| KNN     |     0.9995 |     0.8602 | 0.8612 |        0.0004 |    0.9285 |
| XGB     |     0.9994 |     0.8218 | 0.8218 |        0.0006 |    0.9828 |
| RF      |     0.9995 |     0.8523 | 0.8576 |        0.0004 |    0.953  |


In [ ]:

#start metahurestic
# === Load data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# === Split and scale ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === PSO Feature Selection Class ===
class PSO_FeatureSelection:
    def __init__(self, n_particles, max_iter, fitness_function, n_features, w=0.7, c1=1.5, c2=1.5):
        self.n_particles = n_particles
        self.max_iter = max_iter
        self.fitness_function = fitness_function
        self.n_features = n_features
        self.w = w
        self.c1 = c1
        self.c2 = c2
        self.gbest_score = -float("inf")
        self.gbest_pos = None

    def initialize(self):
        particles = np.random.rand(self.n_particles, self.n_features)
        velocities = np.random.rand(self.n_particles, self.n_features) * 0.1
        pbest_pos = particles.copy()
        pbest_scores = np.full(self.n_particles, -float("inf"))
        return particles, velocities, pbest_pos, pbest_scores

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def binarize(self, particle):
        return (self.sigmoid(particle) > 0.5).astype(int)

    def run(self):
        particles, velocities, pbest_pos, pbest_scores = self.initialize()

        for _ in range(self.max_iter):
            for i in range(self.n_particles):
                binary_particle = self.binarize(particles[i])
                if np.sum(binary_particle) == 0:
                    fitness = -1
                else:
                    fitness = self.fitness_function(binary_particle)
                if fitness > pbest_scores[i]:
                    pbest_scores[i] = fitness
                    pbest_pos[i] = particles[i].copy()
                if fitness > self.gbest_score:
                    self.gbest_score = fitness
                    self.gbest_pos = particles[i].copy()

            for i in range(self.n_particles):
                r1, r2 = np.random.rand(self.n_features), np.random.rand(self.n_features)
                cognitive = self.c1 * r1 * (pbest_pos[i] - particles[i])
                social = self.c2 * r2 * (self.gbest_pos - particles[i])
                velocities[i] = self.w * velocities[i] + cognitive + social
                particles[i] = np.clip(particles[i] + velocities[i], 0, 1)

        return self.binarize(self.gbest_pos)

# === Fitness Function ===
def fitness_function(feature_mask):
    feature_mask = feature_mask.astype(bool)
    if not np.any(feature_mask):
        return 0

    X_sel = X_train[:, feature_mask]
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_sel):
        X_fold_train, X_fold_val = X_sel[train_idx], X_sel[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
        model = LogisticRegression(solver="liblinear")
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        scores.append(f1_score(y_fold_val, y_pred))

    avg_f1 = np.mean(scores)

    # Add penalty for too many features
    feature_ratio = np.sum(feature_mask) / len(feature_mask)
    penalty = 0.1 * feature_ratio
    return avg_f1 - penalty

# === Run PSO Feature Selection ===
print("Running fast PSO feature selection...")
pso = PSO_FeatureSelection(
    n_particles=6,
    max_iter=10,
    fitness_function=fitness_function,
    n_features=X_train.shape[1]
)
best_solution = pso.run()
selected_features = best_solution.astype(bool)
print(f"Selected {np.sum(selected_features)} features")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train[:, selected_features], y_train)
    y_pred = model.predict(X_test[:, selected_features])
    y_prob = model.predict_proba(X_test[:, selected_features])[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_pso = pd.DataFrame(results)
print("\nF(PSO selected features):")
print(results_df_pso.to_markdown(index=False))


Running fast PSO feature selection...
Selected 25 features


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:49:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



F(PSO selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9747 |     0.1109 | 0.2293 |        0.0249 |    0.9719 |
| KNN     |     0.9996 |     0.8804 | 0.8821 |        0.0004 |    0.9336 |
| XGB     |     0.9995 |     0.8586 | 0.8587 |        0.0004 |    0.9692 |
| RF      |     0.9995 |     0.8343 | 0.8401 |        0.0004 |    0.9479 |


In [ ]:

# === Load and preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Simulated Annealing Feature Selection Class ===
class SimulatedAnnealingFeatureSelection:
    def __init__(self, fitness_function, n_features, max_iter=100, initial_temp=100.0, cooling_rate=0.95, min_temp=1e-3):
        self.fitness_function = fitness_function
        self.n_features = n_features
        self.max_iter = max_iter
        self.initial_temp = initial_temp
        self.cooling_rate = cooling_rate
        self.min_temp = min_temp

    def initialize_solution(self):
        return np.random.randint(0, 2, self.n_features)

    def neighbor_solution(self, solution):
        neighbor = solution.copy()
        idx = np.random.randint(0, self.n_features)
        neighbor[idx] = 1 - neighbor[idx]
        return neighbor

    def acceptance_probability(self, old_score, new_score, temperature):
        if new_score > old_score:
            return 1.0
        return np.exp((new_score - old_score) / temperature)

    def run(self):
        current_solution = self.initialize_solution()
        best_solution = current_solution.copy()
        current_score = self.fitness_function(current_solution)
        best_score = current_score
        temperature = self.initial_temp

        for _ in range(self.max_iter):
            neighbor = self.neighbor_solution(current_solution)
            if np.sum(neighbor) == 0:
                continue
            neighbor_score = self.fitness_function(neighbor)

            if self.acceptance_probability(current_score, neighbor_score, temperature) > np.random.rand():
                current_solution = neighbor
                current_score = neighbor_score
                if neighbor_score > best_score:
                    best_score = neighbor_score
                    best_solution = neighbor.copy()

            temperature *= self.cooling_rate
            if temperature < self.min_temp:
                break

        return best_solution.astype(bool)

# === Fitness function ===
def fitness_function(feature_mask):
    feature_mask = feature_mask.astype(bool)
    if not np.any(feature_mask):
        return 0

    X_sel = X_train[:, feature_mask]
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_sel):
        X_fold_train, X_fold_val = X_sel[train_idx], X_sel[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
        model = LogisticRegression(solver="liblinear")
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        scores.append(f1_score(y_fold_val, y_pred))

    avg_f1 = np.mean(scores)
    feature_ratio = np.sum(feature_mask) / len(feature_mask)
    penalty = 0.1 * feature_ratio
    return avg_f1 - penalty

# === Run Simulated Annealing ===
print("Running Simulated Annealing for feature selection...")
sa = SimulatedAnnealingFeatureSelection(
    fitness_function=fitness_function,
    n_features=X_train.shape[1],
    max_iter=100,
    initial_temp=100.0,
    cooling_rate=0.90,
    min_temp=1e-3
)
best_solution = sa.run()
selected_features = best_solution
print(f"Selected {np.sum(selected_features)} features")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "Random Forest": "RF",
    "XGBoost": "XGB"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train[:, selected_features], y_train)
    y_pred = model.predict(X_test[:, selected_features])
    y_prob = model.predict_proba(X_test[:, selected_features])[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_sa = pd.DataFrame(results)
print("\n SA selected features")
print(results_df_sa.to_markdown(index=False))


Running Simulated Annealing for feature selection...
Selected 10 features


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:20:27] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



 SA selected features
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9726 |     0.1013 | 0.2158 |        0.028  |    0.9675 |
| KNN     |     0.9996 |     0.871  | 0.872  |        0.0005 |    0.9234 |
| RF      |     0.9996 |     0.8652 | 0.8694 |        0.0004 |    0.948  |
| XGB     |     0.9995 |     0.8526 | 0.8528 |        0.0005 |    0.9639 |


In [ ]:
# === Load and preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Fitness function ===
def fitness_function(feature_mask):
    feature_mask = feature_mask.astype(bool)
    if not np.any(feature_mask):
        return 0

    X_sel = X_train[:, feature_mask]
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_sel):
        X_fold_train, X_fold_val = X_sel[train_idx], X_sel[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
        model = LogisticRegression(solver="liblinear")
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        scores.append(f1_score(y_fold_val, y_pred))

    return np.mean(scores)

# === Ant Colony Optimization for Feature Selection ===
class AntColonyFeatureSelection:
    def __init__(self, fitness_function, n_features, n_ants=10, max_iter=30, alpha=1.0, beta=1.0, evaporation_rate=0.2, pheromone_init=0.1):
        self.fitness_function = fitness_function
        self.n_features = n_features
        self.n_ants = n_ants
        self.max_iter = max_iter
        self.alpha = alpha
        self.beta = beta
        self.evaporation_rate = evaporation_rate
        self.pheromone = np.ones(n_features) * pheromone_init
        self.best_solution = None
        self.best_score = -np.inf

    def probability(self):
        return self.pheromone / self.pheromone.sum()

    def construct_solution(self):
        solution = np.zeros(self.n_features)
        prob = self.probability()
        for i in range(self.n_features):
            if np.random.rand() < prob[i]:
                solution[i] = 1
        return solution.astype(int)

    def run(self):
        for _ in range(self.max_iter):
            all_solutions = []
            all_scores = []

            for _ in range(self.n_ants):
                candidate = self.construct_solution()
                if np.sum(candidate) == 0:
                    continue
                score = self.fitness_function(candidate)
                all_solutions.append(candidate)
                all_scores.append(score)
                if score > self.best_score:
                    self.best_score = score
                    self.best_solution = candidate.copy()

            # Update pheromones
            self.pheromone *= (1 - self.evaporation_rate)
            for sol, score in zip(all_solutions, all_scores):
                self.pheromone += score * sol

        return self.best_solution.astype(bool)

# === Run ACO Feature Selection ===
print("Running Ant Colony Optimization for feature selection...")
aco = AntColonyFeatureSelection(
    fitness_function=fitness_function,
    n_features=X_train.shape[1],
    n_ants=10,
    max_iter=30,
    alpha=1,
    beta=1,
    evaporation_rate=0.3
)
best_solution = aco.run()
selected_features = best_solution
print(f"Selected {np.sum(selected_features)} features")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "Random Forest": "RF",
    "XGBoost": "XGB"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train[:, selected_features], y_train)
    y_pred = model.predict(X_test[:, selected_features])
    y_prob = model.predict_proba(X_test[:, selected_features])[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_aco = pd.DataFrame(results)
print("\n ACO selected features")
print(results_df_aco.to_markdown(index=False))


Running Ant Colony Optimization for feature selection...
Selected 4 features


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:39:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



 ACO selected features
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9678 |     0.0857 | 0.1946 |        0.0372 |    0.9443 |
| KNN     |     0.9992 |     0.7176 | 0.7258 |        0.0008 |    0.8824 |
| RF      |     0.9992 |     0.7262 | 0.7361 |        0.0006 |    0.9323 |
| XGB     |     0.9979 |     0.553  | 0.5714 |        0.0017 |    0.9409 |


In [ ]:

df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Fitness function ===
def fitness_function(feature_mask):
    feature_mask = feature_mask.astype(bool)
    if not np.any(feature_mask):
        return 0

    X_sel = X_train[:, feature_mask]
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_sel):
        X_fold_train, X_fold_val = X_sel[train_idx], X_sel[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
        model = LogisticRegression(solver="liblinear")
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        scores.append(f1_score(y_fold_val, y_pred))

    return np.mean(scores)

# === Grey Wolf Optimizer for Feature Selection ===
class GWOFeatureSelection:
    def __init__(self, fitness_function, n_features, n_wolves=10, max_iter=30):
        self.fitness_function = fitness_function
        self.n_features = n_features
        self.n_wolves = n_wolves
        self.max_iter = max_iter

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def binarize(self, pos):
        return (self.sigmoid(pos) > 0.5).astype(int)

    def run(self):
        wolves = np.random.uniform(0, 1, (self.n_wolves, self.n_features))
        alpha_pos = np.zeros(self.n_features)
        beta_pos = np.zeros(self.n_features)
        delta_pos = np.zeros(self.n_features)
        alpha_score = beta_score = delta_score = -np.inf

        for t in range(self.max_iter):
            for i in range(self.n_wolves):
                wolf_bin = self.binarize(wolves[i])
                if np.sum(wolf_bin) == 0:
                    continue
                fitness = self.fitness_function(wolf_bin)

                if fitness > alpha_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = alpha_score, alpha_pos.copy()
                    alpha_score, alpha_pos = fitness, wolves[i].copy()
                elif fitness > beta_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = fitness, wolves[i].copy()
                elif fitness > delta_score:
                    delta_score, delta_pos = fitness, wolves[i].copy()

            a = 2 - t * (2 / self.max_iter)
            for i in range(self.n_wolves):
                r1, r2 = np.random.rand(3, self.n_features), np.random.rand(3, self.n_features)
                A1, C1 = 2*a*r1[0] - a, 2*r2[0]
                A2, C2 = 2*a*r1[1] - a, 2*r2[1]
                A3, C3 = 2*a*r1[2] - a, 2*r2[2]

                D_alpha = np.abs(C1 * alpha_pos - wolves[i])
                D_beta = np.abs(C2 * beta_pos - wolves[i])
                D_delta = np.abs(C3 * delta_pos - wolves[i])

                X1 = alpha_pos - A1 * D_alpha
                X2 = beta_pos - A2 * D_beta
                X3 = delta_pos - A3 * D_delta

                wolves[i] = np.clip((X1 + X2 + X3) / 3, 0, 1)

        return self.binarize(alpha_pos).astype(bool)

# === Run GWO Feature Selection ===
print("Running Grey Wolf Optimization for feature selection...")
gwo = GWOFeatureSelection(
    fitness_function=fitness_function,
    n_features=X_train.shape[1],
    n_wolves=10,
    max_iter=30
)
best_solution = gwo.run()
selected_features = best_solution
print(f"Selected {np.sum(selected_features)} features")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train)-sum(y_train))/sum(y_train),
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "Random Forest": "RF",
    "XGBoost": "XGB"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train[:, selected_features], y_train)
    y_pred = model.predict(X_test[:, selected_features])
    y_prob = model.predict_proba(X_test[:, selected_features])[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_gwo = pd.DataFrame(results)
print("\n GWO selected features")
print(results_df_gwo.to_markdown(index=False))


Running Grey Wolf Optimization for feature selection...
Selected 28 features


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [14:01:20] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



 GWO selected features
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9755 |     0.1142 | 0.233  |        0.0237 |    0.9709 |
| KNN     |     0.9996 |     0.8757 | 0.877  |        0.0004 |    0.9336 |
| RF      |     0.9995 |     0.8523 | 0.8576 |        0.0004 |    0.9479 |
| XGB     |     0.9996 |     0.8691 | 0.8692 |        0.0004 |    0.975  |


In [ ]:
# === Load and preprocess data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === CSO Fitness Function ===
@optunity.cross_validated(x=X_train, y=y_train, num_folds=3)
def feature_selection_cso(x_train, y_train, x_test, y_test, **kwargs):
    feature_mask = np.array([kwargs[f"f{i}"] for i in range(X_train.shape[1])]) > 0.5
    if sum(feature_mask) == 0:
        return 0
    X_train_sel = x_train[:, feature_mask]
    X_test_sel = x_test[:, feature_mask]
    model = LogisticRegression(solver="liblinear")
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    return f1_score(y_test, y_pred)

# === Define CSO Search Space ===
search_space = {f"f{i}": [0, 1] for i in range(X_train.shape[1])}

# === Run CSO Optimization ===
print("Running Cuckoo Search Optimization (CSO) for feature selection...")
optimal_config, details, _ = optunity.maximize(feature_selection_cso, num_evals=50, **search_space)

# === Get Selected Features ===
selected_features = np.array([optimal_config[f"f{i}"] for i in range(X_train.shape[1])]) > 0.5
X_train_sel = X_train[:, selected_features]
X_test_sel = X_test[:, selected_features]
print(f"Selected {np.sum(selected_features)} features")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "Random Forest": "RF",
    "XGBoost": "XGB"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    y_prob = model.predict_proba(X_test_sel)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Clean Results Table ===
results_df_cso = pd.DataFrame(results)
print("\n CSO selected features")
print(results_df_cso.to_markdown(index=False))


Running Cuckoo Search Optimization (CSO) for feature selection...
Selected 19 features


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [14:24:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



 CSO selected features
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9738 |     0.1077 | 0.2257 |        0.0268 |    0.9716 |
| KNN     |     0.9996 |     0.8852 | 0.8873 |        0.0004 |    0.9387 |
| RF      |     0.9995 |     0.8475 | 0.8522 |        0.0004 |    0.958  |
| XGB     |     0.9995 |     0.8421 | 0.8423 |        0.0005 |    0.9659 |


In [ ]:
!pip install deap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 3.8 MB/s eta 0:00:00


In [ ]:

from deap import base, creator, tools, algorithms
import random
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# GA Settings
n_features = X_train.shape[1]
POP_SIZE = 20
N_GEN = 20
CXPB = 0.5
MUTPB = 0.2

# Define Fitness Function (maximize F1-score)
def evaluate(individual):
    selected = np.array(individual, dtype=bool)
    if not selected.any():
        return 0,
    X_selected = X_train[:, selected]
    clf = LogisticRegression(solver='liblinear')
    f1 = cross_val_score(clf, X_selected, y_train, cv=3, scoring='f1').mean()
    return f1,

# GA Setup using DEAP
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", lambda: random.randint(0, 1))
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
random.seed(42)
pop = toolbox.population(n=POP_SIZE)
algorithms.eaSimple(pop, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=N_GEN, verbose=False)

# Select best individual
best_ind = tools.selBest(pop, k=1)[0]
selected_features = np.array(best_ind, dtype=bool)
print(f"✅ GA selected {np.sum(selected_features)} features.")

X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

# Define models including Random Forest
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", n_jobs=-1)
}

# Train & Evaluate
results = []
for name, model in models.items():
    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)
    y_prob = model.predict_proba(X_test_selected)[:, 1]

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        matthews_corrcoef(y_test, y_pred),
        brier_score_loss(y_test, y_prob),
        roc_auc_score(y_test, y_prob)
    ])

# Results Table
results_df_ga = pd.DataFrame(results, columns=["Model", "Accuracy", "F1 Score", "MCC", "Brier Score", "AUC-ROC"])
print("\n✅ Model Performance with GA + Feature Selection:")
print(results_df_ga.to_markdown(index=False))


✅ GA selected 22 features.

✅ Model Performance with GA + Feature Selection:
| Model               |   Accuracy |   F1 Score |      MCC |   Brier Score |   AUC-ROC |
|:--------------------|-----------:|-----------:|---------:|--------------:|----------:|
| Logistic Regression |   0.999087 |   0.704545 | 0.708699 |   0.00078222  |  0.955456 |
| K-Nearest Neighbors |   0.999561 |   0.863388 | 0.865363 |   0.00040448  |  0.943748 |
| Random Forest       |   0.999526 |   0.847458 | 0.852162 |   0.000408174 |  0.952927 |
| XGBoost             |   0.999508 |   0.849462 | 0.850451 |   0.000424251 |  0.977087 |


In [ ]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set Plot Style
sns.set(style="whitegrid")

# Step 1: Add Feature Selection labels
results_df_pso["Feature Selection"] = "PSO"  # Particle Swarm Optimization
results_df_sa["Feature Selection"] = "Simulated Annealing"
results_df_aco["Feature Selection"] = "Ant Colony Optimization"
results_df_cso["Feature Selection"] = "Crow Search Optimization"
results_df_ga["Feature Selection"] = "Genetic Algorithm"
results_df_gwo["Feature Selection"] = "Grey Wolf Optimization"

# Step 2: Combine all results into one DataFrame
combined_results = pd.concat([
    results_df_pso,
    results_df_sa,
    results_df_aco,
    results_df_cso,
    results_df_ga,
    results_df_gwo
], ignore_index=True)

# Step 3: Define a function to plot any metric
def plot_metric(ax, metric_name, palette="viridis", ylim=(0.0, 1.0)):
    sns.barplot(
        data=combined_results,
        x="Model",
        y=metric_name,
        hue="Feature Selection",
        palette=palette,
        ax=ax
    )
    ax.set_title(f"{metric_name} Comparison", fontsize=14, pad=15)  # Added some padding
    ax.set_xlabel("Model", fontsize=12)
    ax.set_ylabel(metric_name, fontsize=12)
    ax.set_ylim(*ylim)
    ax.tick_params(axis='x', rotation=45)
    ax.legend_.remove()  # Remove legend from individual plots

# Step 4: Create a 5x1 grid of plots
fig, axes = plt.subplots(5, 1, figsize=(14, 28))  # 5 rows, 1 column

fig.suptitle('Comparison of Different Feature Selection Methods Across Metrics', fontsize=20, y=1.02)  # Adjusted title position

# Plot each metric
plot_metric(axes[0], "Accuracy", palette="viridis", ylim=(0.90, 1.01))
plot_metric(axes[1], "F1 Score", palette="plasma", ylim=(0.80, 1.01))
plot_metric(axes[2], "MCC", palette="mako", ylim=(0.0, 1.0))
plot_metric(axes[3], "Balanced Accuracy", palette="rocket", ylim=(0.0, 1.0))
plot_metric(axes[4], "AUC ROC", palette="coolwarm", ylim=(0.5, 1.0))

# Step 5: Create a single legend for all plots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Feature Selection", loc='lower center', ncol=3, fontsize=12, frameon=False)

# Step 6: Adjust spacing
plt.subplots_adjust(hspace=0.5, top=0.95, bottom=0.08)  # hspace increased between plots

# Step 7: Show plot
plt.show()


NameError: name 'results_df_sa' is not defined

without Feature selection code

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, brier_score_loss, roc_auc_score

# Load and preprocess data
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    )
}

# Evaluate models
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Brier Score": brier_score_loss(y_test, y_prob),
        "AUC-ROC": roc_auc_score(y_test, y_prob),
        "Features Used": X_train.shape[1]
    })

# Output results
results_df = pd.DataFrame(results)
print("\nModel Performance Without Feature Selection:")
print(results_df.to_markdown(index=False))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [05:06:38] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Model Performance Without Feature Selection:
| Model               |   Accuracy |   F1 Score |      MCC |   Brier Score |   AUC-ROC |   Features Used |
|:--------------------|-----------:|-----------:|---------:|--------------:|----------:|----------------:|
| Logistic Regression |   0.975545 |   0.114431 | 0.233283 |   0.0235108   |  0.972093 |              30 |
| K-Nearest Neighbors |   0.999579 |   0.870968 | 0.872023 |   0.000414434 |  0.933586 |              30 |
| Random Forest       |   0.999508 |   0.83908  | 0.845645 |   0.000409613 |  0.952905 |              30 |
| XGBoost             |   0.999526 |   0.858639 | 0.858697 |   0.00042807  |  0.968238 |              30 |


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# === Load data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# === Correlation Coefficient Calculation ===
correlation_scores = X.corrwith(y).abs()  # Pearson correlation with target
top_k = 15
top_features = correlation_scores.sort_values(ascending=False).head(top_k).index
X = X[top_features]

# === Train/Test Split & Scale ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Selected {len(top_features)} features based on Correlation Coefficient.")

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Results ===
results_df_corr = pd.DataFrame(results)
print("\n📊 Model Performance (Correlation Coefficient selected features):")
print(results_df_corr.to_markdown(index=False))


Selected 15 features based on Correlation Coefficient.


/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [07:12:21] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Model Performance (Correlation Coefficient selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9732 |     0.1056 | 0.2233 |        0.027  |    0.9699 |
| KNN     |     0.9995 |     0.8602 | 0.8612 |        0.0004 |    0.9285 |
| XGB     |     0.9994 |     0.8218 | 0.8218 |        0.0006 |    0.9828 |
| RF      |     0.9995 |     0.8475 | 0.8522 |        0.0004 |    0.9582 |


In [ ]:
# === Imports ===
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    brier_score_loss, roc_auc_score
)

# === Load data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# === Variance Threshold Feature Selection ===
# Set a threshold (default is 0 which removes constant features)
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X)

# Get selected feature names
selected_columns = X.columns[selector.get_support()]
X = pd.DataFrame(X_var, columns=selected_columns)

print(f"Selected {X.shape[1]} features based on Variance Threshold.")

# === Train/Test Split & Scale ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Define Models ===
models = {
    "Logistic Regression": LogisticRegression(solver="liblinear", class_weight="balanced"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1),
    "XGBoost": XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# === Short Model Names ===
model_name_map = {
    "Logistic Regression": "LR",
    "K-Nearest Neighbors": "KNN",
    "XGBoost": "XGB",
    "Random Forest": "RF"
}

# === Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Results ===
results_df_variance = pd.DataFrame(results)
print("\n📊 Model Performance (Variance Threshold selected features):")
print(results_df_variance.to_markdown(index=False))


Selected 30 features based on Variance Threshold.


/usr/local/lib/python3.11/dist-packages/xgboost/training.py:183: UserWarning: [07:25:49] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



📊 Model Performance (Variance Threshold selected features):
| Model   |   Accuracy |   F1 Score |    MCC |   Brier Score |   AUC-ROC |
|:--------|-----------:|-----------:|-------:|--------------:|----------:|
| LR      |     0.9755 |     0.1144 | 0.2333 |        0.0235 |    0.9721 |
| KNN     |     0.9996 |     0.871  | 0.872  |        0.0004 |    0.9336 |
| XGB     |     0.9995 |     0.8586 | 0.8587 |        0.0004 |    0.9682 |
| RF      |     0.9995 |     0.8391 | 0.8456 |        0.0004 |    0.9529 |


In [ ]:
# === Imports (reuse previous) ===
from sklearn.feature_selection import SelectFromModel

# === Load data ===
df = pd.read_csv('/content/drive/MyDrive/Research /creditcard.csv')
X = df.drop(columns=["Class"])
y = df["Class"]

# === LASSO Feature Selection ===
lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.01, class_weight="balanced")
lasso.fit(X, y)

selector = SelectFromModel(lasso, prefit=True)
X_lasso = selector.transform(X)
selected_columns = X.columns[selector.get_support()]
X = pd.DataFrame(X_lasso, columns=selected_columns)

print(f"Selected {X.shape[1]} features using LASSO (L1).")

# === Train/Test Split & Scale ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# === Define and Evaluate Models ===
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name_map[name],
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
        "Brier Score": round(brier_score_loss(y_test, y_prob), 4),
        "AUC-ROC": round(roc_auc_score(y_test, y_prob), 4)
    })

# === Output Results ===
results_df_lasso = pd.DataFrame(results)
print("\n📊 Model Performance (LASSO selected features):")
print(results_df_lasso.to_markdown(index=False))
